# Our PR2 MJCF in the Multiverse apartment

This notebook uses repository-local PR2, apartment, milk-box, and cereal-box MJCF resources. It adapts the parsed PR2 planar base to an SDT `OmniDrive`, then executes standard PyCRAM park-arms, torso, navigation, and right-arm pickup actions through Giskard while `MujocoSynchronizer` mirrors state and attachment changes into MuJoCo.

The scene starts with parked arms, a low torso, and closed kitchen drawers. The free milk and cereal boxes begin above the right countertop and visibly fall onto it under MuJoCo gravity and contacts.


In [ ]:
import os
import threading
import time
from dataclasses import dataclass
from pathlib import Path
from xml.etree import ElementTree as ET

%matplotlib inline
import matplotlib.pyplot as plt
import mujoco
import numpy as np
import rclpy
from IPython.display import display
from rclpy.executors import SingleThreadedExecutor

from physics_simulators.mujoco_simulator import MujocoSimulator
from pycram.datastructures.dataclasses import Context
from pycram.datastructures.enums import (
    ApproachDirection,
    Arms,
    VerticalAlignment,
)
from pycram.datastructures.grasp import GraspDescription
from pycram.motion_executor import simulated_robot
from pycram.plans.factories import sequential
from pycram.robot_plans.actions.core.navigation import NavigateAction
from pycram.robot_plans.actions.core.pick_up import PickUpAction
from pycram.robot_plans.actions.core.robot_body import MoveTorsoAction, ParkArmsAction
from semantic_digital_twin.adapters.mjcf import MJCFParser
from semantic_digital_twin.adapters.ros.visualization.viz_marker import (
    VizMarkerPublisher,
)
from semantic_digital_twin.adapters.multi_sim import MujocoBuilder, MujocoSynchronizer
from semantic_digital_twin.callbacks.callback import StateChangeCallback
from semantic_digital_twin.datastructures.definitions import StaticJointState, TorsoState
from semantic_digital_twin.robots.loaders import load_pr2_mjcf
from semantic_digital_twin.robots.pr2 import PR2
from semantic_digital_twin.semantic_annotations.semantic_annotations import Cereal, Milk
from semantic_digital_twin.spatial_types.spatial_types import HomogeneousTransformationMatrix, Pose
from semantic_digital_twin.world_description.connections import (
    ActiveConnection1DOF,
    FixedConnection,
)
from semantic_digital_twin.world_description.geometry import Box, Color, Scale
from semantic_digital_twin.world_description.shape_collection import ShapeCollection


def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'semantic_digital_twin/resources/mujoco_resources').is_dir():
            return candidate
    raise FileNotFoundError('Run this notebook from inside the repository.')


REPO_ROOT = find_repo_root()
MUJOCO_RESOURCES = REPO_ROOT / 'semantic_digital_twin/resources/mujoco_resources'
PR2_MJCF = (MUJOCO_RESOURCES / 'robots/pr2/pr2.xml').resolve()
APARTMENT_MJCF = (MUJOCO_RESOURCES / 'environments/apartment/apartment.xml').resolve()
MILK_MJCF = (MUJOCO_RESOURCES / 'objects/milk_box/milk_box.xml').resolve()
CEREAL_MJCF = (MUJOCO_RESOURCES / 'objects/cereal_box/cereal_box.xml').resolve()
GENERATED_SCENE = Path('/tmp/pr2_multiverse_apartment.xml')
HEADLESS = os.environ.get('MUJOCO_HEADLESS', '0') == '1'

# Face the right countertop while keeping parked grippers clear of cabinets.
PR2_X = 1.6
PR2_Y = 1.875
PR2_YAW = 0.0

for resource in (PR2_MJCF, APARTMENT_MJCF, MILK_MJCF, CEREAL_MJCF):
    if not resource.is_file():
        raise FileNotFoundError(resource)
print(f'PR2:       {PR2_MJCF}')
print(f'Apartment: {APARTMENT_MJCF}')
print(f'Milk:      {MILK_MJCF}')
print(f'Cereal:    {CEREAL_MJCF}')
print(f'Output:    {GENERATED_SCENE}')
print(f'Headless:  {HEADLESS}')


## World Setup

This cell creates the complete SDT world. It loads the PR2 through the PR2-specific MJCF loader, parses and merges the apartment and free objects, and attaches the PR2, milk, and cereal semantic annotations used by PyCRAM and Giskard.


In [ ]:
pr2_world = load_pr2_mjcf(
    PR2_MJCF,
    x=PR2_X,
    y=PR2_Y,
    yaw=PR2_YAW,
)

apartment_parser = MJCFParser(str(APARTMENT_MJCF))
# Each parser otherwise creates a root body named 'world'.
apartment_parser.spec.worldbody.name = 'apartment_world'
apartment_world = apartment_parser.parse()

pr2_world.merge_world(
    apartment_world,
    root_connection=FixedConnection(
        parent=pr2_world.root, child=apartment_world.root
    ),
)
world = pr2_world

# Start visibly above the countertop so viewer startup demonstrates gravity.
OBJECT_INITIAL_POSES = {
    'milk_box': (3.0, 1.70, 2.70),
    'cereal_box': (3.0, 2.05, 2.72),
}


def merge_free_mjcf_object(mjcf_path, world_root_name, body_name, pose):
    parser = MJCFParser(str(mjcf_path))
    parser.spec.worldbody.name = world_root_name
    object_world = parser.parse()
    object_world_root = object_world.root
    world.merge_world(
        object_world,
        root_connection=FixedConnection(parent=world.root, child=object_world_root),
    )
    object_body = world.get_body_by_name(body_name)
    with world.modify_world():
        world.move_branch(object_body, world.root)
    object_body.parent_connection.origin = pose
    return object_body


milk_body = merge_free_mjcf_object(
    MILK_MJCF,
    'milk_world',
    'milk_box',
    HomogeneousTransformationMatrix.from_xyz_rpy(
        *OBJECT_INITIAL_POSES['milk_box'], reference_frame=world.root
    ),
)
cereal_body = merge_free_mjcf_object(
    CEREAL_MJCF,
    'cereal_world',
    'cereal_box',
    HomogeneousTransformationMatrix.from_xyz_rpy(
        *OBJECT_INITIAL_POSES['cereal_box'], reference_frame=world.root
    ),
)



drive = world.get_connection_by_name('odom_combined_T_base_footprint')
print(f'{len(world.bodies)} bodies')
print(f'{len(world.connections)} connections')
print(f'{len(world.actuators)} parsed joint actuators')

with world.modify_world():
    pr2 = PR2._init_empty_robot(world)
    pr2._setup_semantic_annotations()
    pr2._setup_velocity_limits()
    pr2._setup_hardware_interfaces()
    pr2._setup_joint_states()
    world.add_semantic_annotation(pr2)

    milk = Milk(root=milk_body)
    cereal = Cereal(root=cereal_body)
    world.add_semantic_annotation(milk)
    world.add_semantic_annotation(cereal)

    # The source MJCF has visual meshes but no collision shape on base_link.
    base_link = world.get_body_by_name('base_link')
    base_link.collision = ShapeCollection(
        shapes=[
            Box(
                scale=Scale(0.68, 0.68, 0.25),
                origin=HomogeneousTransformationMatrix.from_xyz_rpy(
                    z=0.12, reference_frame=base_link
                ),
                color=Color(0.0, 0.0, 0.0, 0.0),
            )
        ],
        reference_frame=base_link,
    )

context = Context(world=world, robot=pr2)
context.evaluate_conditions = True
assert pr2.drive is drive and pr2.drive.has_hardware_interface
print(f'Objects: {milk.root.name.name}, {cereal.root.name.name}')
print(f'Controlled drive: {pr2.drive.name.name}')


## World Initialization

This cell defines the deterministic starting state: parked PR2 arms, low torso, neutral head camera, closed apartment drawers and doors, and the initial mobile-base pose supplied by the PR2 loader.


In [ ]:
initial_joint_targets = {}
for arm in pr2.arms:
    parked = arm.get_joint_state_by_type(StaticJointState.PARK)
    initial_joint_targets.update(
        {connection.name.name: value for connection, value in parked.items()}
    )
initial_joint_targets.update(
    {
        connection.name.name: value
        for connection, value in pr2.torso.get_joint_state_by_type(TorsoState.LOW).items()
    }
)

# The merged MJCF keyframe can contain stale head values. Keep the camera
# centered and nearly horizontal in the robot-forward direction.
initial_joint_targets.update(
    {
        'head_pan_joint': 0.0,
        'head_tilt_joint': 0.0,
    }
)

# The source kitchen has zero as the closed position for every drawer and door.
closed_apartment_joints = [
    connection.name.name
    for connection in world.connections
    if isinstance(connection, ActiveConnection1DOF)
    and ('drawer' in connection.name.name or 'door' in connection.name.name)
]
initial_joint_targets.update({name: 0.0 for name in closed_apartment_joints})
for name, value in initial_joint_targets.items():
    world.get_connection_by_name(name).position = value
world.notify_state_change()
print(f'Initialized {len(closed_apartment_joints)} drawer/door joints as closed.')


## RViz2 visualization

This cell publishes the live SDT world as a transient-local `MarkerArray` and publishes its kinematic state on `/tf`. Keep this notebook kernel running, start `rviz2` in a ROS2-sourced terminal, set the fixed frame printed below, and add a `MarkerArray` display for `/semworld/viz_marker` with transient-local durability.


In [ ]:
# Stop an older notebook-owned ROS publisher before rerunning this cell.
if 'rviz_marker_publisher' in globals():
    if rviz_marker_publisher._tf_publisher is not None:
        rviz_marker_publisher._tf_publisher.stop()
    rviz_marker_publisher.stop()
if 'rviz_executor' in globals():
    rviz_executor.shutdown(timeout_sec=1.0)
if 'rviz_executor_thread' in globals() and rviz_executor_thread.is_alive():
    rviz_executor_thread.join(timeout=1.0)
if 'rviz_node' in globals():
    rviz_node.destroy_node()
if globals().get('rviz_owns_rclpy', False) and rclpy.ok():
    rclpy.shutdown()

rviz_owns_rclpy = not rclpy.ok()
if rviz_owns_rclpy:
    rclpy.init()

rviz_node = rclpy.create_node('pr2_apartment_sdt_visualization')
rviz_executor = SingleThreadedExecutor()
rviz_executor.add_node(rviz_node)
rviz_executor_thread = threading.Thread(
    target=rviz_executor.spin,
    daemon=True,
    name='pr2-apartment-rviz-executor',
)
rviz_executor_thread.start()




In [ ]:
rviz_marker_publisher = VizMarkerPublisher(_world=world, node=rviz_node)
rviz_marker_publisher.with_tf_publisher()
rviz_fixed_frame = str(world.root.name)
print(f'RViz fixed frame: {rviz_fixed_frame}')
print('RViz MarkerArray topic: /semworld/viz_marker')

## MuJoCo Scene Build

This cell converts the initialized SDT world into one MJCF scene. It configures MuJoCo stability, gravity, lighting, cameras, framebuffer size, mobile-base actuators, and holding actuators for apartment furniture.


In [ ]:
class BalancedMujocoBuilder(MujocoBuilder):
    def _start_build(self, file_path):
        super()._start_build(file_path)
        self.spec.compiler.balanceinertia = True
        self.spec.compiler.boundmass = 1e-6
        self.spec.compiler.boundinertia = 1e-6


BalancedMujocoBuilder().build_world(world, str(GENERATED_SCENE))

tree = ET.parse(GENERATED_SCENE)
root = tree.getroot()
compiler = root.find('compiler')
compiler.set('balanceinertia', 'true')
compiler.set('boundmass', '0.000001')
compiler.set('boundinertia', '0.000001')

option = root.find('option')
if option is None:
    option = ET.Element('option')
    root.insert(1, option)
option.set('gravity', '0 0 -9.81')
option.set('integrator', 'implicitfast')

asset = root.find('asset')
visual = root.find('visual')
if visual is None:
    visual = ET.Element('visual')
    root.insert(list(root).index(asset), visual)
ET.SubElement(
    visual,
    'headlight',
    diffuse='0.7 0.7 0.7',
    ambient='0.35 0.35 0.35',
    specular='0.1 0.1 0.1',
)
ET.SubElement(visual, 'rgba', haze='0.15 0.25 0.35 1')
ET.SubElement(
    visual,
    'global',
    azimuth='120',
    elevation='-20',
    offwidth='1280',
    offheight='720',
)
ET.SubElement(
    asset,
    'texture',
    name='scene_skybox',
    type='skybox',
    builtin='gradient',
    rgb1='0.3 0.5 0.7',
    rgb2='0 0 0',
    width='512',
    height='3072',
)

worldbody = root.find('worldbody')
ET.SubElement(
    worldbody,
    'light',
    name='ceiling_light',
    pos='0 0 6',
    dir='0 0 -1',
    directional='true',
    diffuse='0.8 0.8 0.8',
    specular='0.2 0.2 0.2',
)
ET.SubElement(
    worldbody,
    'light',
    name='fill_light',
    pos='2 -2 4',
    dir='-0.3 0.3 -1',
    directional='true',
    diffuse='0.45 0.45 0.45',
    specular='0.1 0.1 0.1',
    castshadow='false',
)

actuator = root.find('actuator')
if actuator is None:
    actuator = ET.SubElement(root, 'actuator')
for name, suffix, kp, ctrlrange in (
    ('base_x_motor', 'x', '10000', '-100 100'),
    ('base_y_motor', 'y', '10000', '-100 100'),
    ('base_yaw_motor', 'yaw', '5000', '-25.1327412287 25.1327412287'),
):
    ET.SubElement(
        actuator,
        'position',
        name=name,
        joint=f'{drive.name.name}_{suffix}',
        kp=kp,
        ctrllimited='true',
        ctrlrange=ctrlrange,
        forcelimited='true',
        forcerange='-1000000 1000000',
    )

camera_body = root.find(".//body[@name='wide_stereo_optical_frame']")
if camera_body is None:
    raise RuntimeError('PR2 wide-stereo optical frame is missing from the built MJCF.')
ET.SubElement(
    camera_body,
    'camera',
    name='pr2_egocentric',
    quat='0 1 0 0',
    fovy='80',
)
ET.SubElement(
      camera_body,
      'camera',
      name='pr2_third_person',
      pos='0 -0.45 -0.9',
      quat='0.04 0.099 0 0',
      fovy='70',
  )


# Keep passive apartment furniture closed while gravity and contacts are active.
for joint_name in closed_apartment_joints:
    ET.SubElement(
        actuator,
        'position',
        name=f'{joint_name}_closed_motor',
        joint=joint_name,
        kp='1000',
        forcelimited='true',
        forcerange='-100000 100000',
    )

ET.indent(tree, space='  ')
tree.write(GENERATED_SCENE, encoding='utf-8', xml_declaration=True)
model = mujoco.MjModel.from_xml_path(str(GENERATED_SCENE))
print(
    f'Compiled: {model.nbody} bodies, {model.njnt} joints, '
    f'{model.nu} actuators, {model.ngeom} geoms, {model.nlight} lights'
)


## Launch the viewer and demonstrate gravity

This starts MuJoCo with gravity and contacts enabled. PR2 begins in front of and facing the milk and cereal without touching the counter. The boxes fall visibly, and `capture_pr2_egocentric()` returns and displays an RGB image from PR2's wide-stereo optical frame. A state-change callback then refreshes MuJoCo as Giskard updates the SDT world.


In [ ]:
simulator = MujocoSimulator(
    file_path=str(GENERATED_SCENE),
    _headless=HEADLESS,
    _step_size=0.001,
    config={
        'gravity': True,
        'contact': True,
        'integrator': mujoco.mjtIntegrator.mjINT_IMPLICITFAST,
    },
)
simulator.set_joints_values(initial_joint_targets)

drive_joint_targets = {
    f'{drive.name.name}_x': PR2_X,
    f'{drive.name.name}_y': PR2_Y,
    f'{drive.name.name}_yaw': PR2_YAW,
}
simulator.set_joints_values(drive_joint_targets)

# Free joints are emitted before the articulated joints in MuJoCo qpos, while
# the merged-world keyframe is serialized in world order. Initialize them by
# joint address before replacing that stale generated keyframe with `home`.
for body_name, xyz in OBJECT_INITIAL_POSES.items():
    body_id = mujoco.mj_name2id(
        simulator._mj_model, mujoco.mjtObj.mjOBJ_BODY, body_name
    )
    joint_id = int(simulator._mj_model.body_jntadr[body_id])
    qpos_address = int(simulator._mj_model.jnt_qposadr[joint_id])
    simulator._mj_data.qpos[qpos_address:qpos_address + 7] = (
        *xyz,
        1.0,
        0.0,
        0.0,
        0.0,
    )
mujoco.mj_forward(simulator._mj_model, simulator._mj_data)


def hold_position_actuators_at_qpos():
    model = simulator._mj_model
    data = simulator._mj_data
    for actuator_index in range(model.nu):
        if model.actuator_trntype[actuator_index] != mujoco.mjtTrn.mjTRN_JOINT:
            continue
        joint_id = int(model.actuator_trnid[actuator_index, 0])
        if joint_id < 0:
            continue
        qpos_address = int(model.jnt_qposadr[joint_id])
        data.ctrl[actuator_index] = data.qpos[qpos_address]


hold_position_actuators_at_qpos()
simulator.save(key_name='home')
synchronizer = MujocoSynchronizer(_world=world, simulator=simulator)
simulator.start(simulate_in_thread=False)

if not HEADLESS:
      viewer = simulator.renderer

      with viewer.lock():
          viewer.cam.type = mujoco.mjtCamera.mjCAMERA_FREE
          viewer.cam.lookat[:] = [2.2, 1.875, 1.0]
          viewer.cam.distance = 5.0
          viewer.cam.azimuth = 120
          viewer.cam.elevation = -20

      viewer.sync()


# Advance two seconds of physics. In the interactive viewer this is paced
# close to real time so the boxes can be seen falling onto the counter.
GRAVITY_DEMO_SECONDS = 2.0
SIMULATION_STEP_SIZE = 0.001
RENDER_EVERY_STEPS = 10
gravity_demo_steps = int(GRAVITY_DEMO_SECONDS / SIMULATION_STEP_SIZE)
for step in range(gravity_demo_steps):
    simulator.step()
    if step % RENDER_EVERY_STEPS == 0:
        simulator.renderer.sync()
        if not simulator.headless:
            time.sleep(SIMULATION_STEP_SIZE * RENDER_EVERY_STEPS)

# Gravity can slightly move passive furniture and articulated robot joints.
# Restore all deterministic robot/furniture targets while leaving free objects
# at the poses where they landed.
simulator.set_joints_values(initial_joint_targets)
hold_position_actuators_at_qpos()
mujoco.mj_forward(simulator._mj_model, simulator._mj_data)
simulator.renderer.sync()
synchronizer._last_sync_time = 0.0
synchronizer._sim_to_world()


@dataclass(eq=False)
class MujocoControlAndRenderCallback(StateChangeCallback):
    simulator: MujocoSimulator
    control_period: float = 1.0 / 50.0

    def _notify(self, **kwargs):
        hold_position_actuators_at_qpos()
        mujoco.mj_forward(self.simulator._mj_model, self.simulator._mj_data)
        self.simulator.renderer.sync()
        if not self.simulator.headless:
            time.sleep(self.control_period)
        self.update_previous_world_state()


mujoco_control_callback = MujocoControlAndRenderCallback(
    _world=world,
    simulator=simulator,
)


def capture_pr2_egocentric(height=720, width=1280, show=True):
    rgb = simulator.capture_rgb(
        camera_name='pr2_egocentric',
        height=height,
        width=width,
    ).result
    if show:
        figure, axes = plt.subplots(figsize=(16, 9))
        axes.imshow(rgb)
        axes.set_title('PR2 egocentric camera')
        axes.axis('off')
        display(figure)
        plt.close(figure)
    return rgb
def capture_pr2_third_person_view(height=720, width=1280, show=True):
    third_person_rgb = simulator.capture_rgb(
      camera_name='pr2_third_person',
      height=720,
      width=1280,
  ).result
    if show:
        figure, axes = plt.subplots(figsize=(16, 9))
        axes.imshow(third_person_rgb)
        axes.set_title('PR2 third-person camera')
        axes.axis('off')
        display(figure)
        plt.close(figure)
    return third_person_rgb


egocentric_rgb = capture_pr2_egocentric(show=not HEADLESS)
third_person_rgb = capture_pr2_third_person_view(show=not HEADLESS)


print(
    'PR2 apartment scene started facing the counter; '
    'milk and cereal fell under gravity.'
)


## Standard PyCRAM action plan

No notebook-local action descriptions are needed. The plan below uses the same stock actions used by PyCRAM task plans, with condition evaluation enabled.


In [ ]:
# Move a short distance toward the counter while preserving the facing direction.
NAVIGATION_GOAL = Pose.from_xyz_rpy(
    x=1.75,
    y=1.875,
    yaw=0.0,
    reference_frame=world.root,
)
simple_plan = sequential(
    [
        ParkArmsAction(Arms.BOTH),
        MoveTorsoAction(TorsoState.HIGH),
        NavigateAction(NAVIGATION_GOAL, keep_joint_states=True),
    ],
    context=context,
).plan
print('Plan: park arms -> raise torso -> move forward -> stop')


## Execute the plan

Run this cell when you are ready to execute the standard PyCRAM actions through Giskard. `MujocoSynchronizer` copies each resulting SDT state update into MuJoCo.


In [ ]:
with simulated_robot:
    simple_plan.perform()

drive_state = world.state
print(
    'Plan finished:',
    f"torso={world.get_connection_by_name('torso_lift_joint').position:.3f}",
    f"base=({drive_state[pr2.drive.x.id].position:.3f}, "
    f"{drive_state[pr2.drive.y.id].position:.3f}, "
    f"yaw={drive_state[pr2.drive.yaw.id].position:.3f})",
)


## Right-arm milk pickup plan

This demo uses the stock PyCRAM `PickUpAction`. The base stance and grasp are deliberately hardcoded for the settled milk pose on this countertop. The action opens the right gripper, reaches the milk, closes the gripper, attaches the milk to `r_gripper_tool_frame`, and lifts it.


In [ ]:
PICKUP_STANCE = Pose.from_xyz_rpy(
    x=2.05,
    y=1.875,
    yaw=0.0,
    reference_frame=world.root,
)
milk_pickup_grasp = GraspDescription(
    approach_direction=ApproachDirection.FRONT,
    vertical_alignment=VerticalAlignment.NoAlignment,
    manipulator=pr2.right_arm.manipulator,
    rotate_gripper=False,
    manipulation_offset=0.08,
)
milk_pickup_plan = sequential(
    [
        ParkArmsAction(Arms.BOTH),
        NavigateAction(PICKUP_STANCE, keep_joint_states=True),
        PickUpAction(
            object_designator=milk.root,
            arm=Arms.RIGHT,
            grasp_description=milk_pickup_grasp,
        ),
    ],
    context=context,
).plan
print('Pickup plan: park arms -> approach counter -> pick and lift milk')


## Execute the milk pickup

The reduced PR2 MJCF omits cosmetic links still named in the full PR2 SRDF, so PyCRAM condition evaluation cannot clone and validate this robot yet. This cell disables condition evaluation only while running the hardcoded demo; it still executes the real `PickUpAction`, Giskard motions, gripper commands, SDT attachment, and MuJoCo synchronization.


In [ ]:
evaluate_conditions_before_pickup = context.evaluate_conditions
context.evaluate_conditions = False
try:
    with simulated_robot:
        milk_pickup_plan.perform()
finally:
    context.evaluate_conditions = evaluate_conditions_before_pickup

milk_parent = milk.root.parent_connection.parent
assert milk_parent is pr2.right_arm.manipulator.tool_frame
simulator.renderer.sync()
print(
    'Milk pickup finished:',
    f'parent={milk_parent.name.name}',
    f'position={milk.root.global_pose.to_position().to_np()[:3]}',
)


In [ ]:
ego = capture_pr2_egocentric()
tpv = capture_pr2_third_person_view()

## Stop before rerunning


In [ ]:
if 'mujoco_control_callback' in globals():
    mujoco_control_callback.stop()
if 'synchronizer' in globals():
    synchronizer.stop()
if 'simulator' in globals():
    simulator.stop()
if 'rviz_marker_publisher' in globals():
    if rviz_marker_publisher._tf_publisher is not None:
        rviz_marker_publisher._tf_publisher.stop()
    rviz_marker_publisher.stop()
if 'rviz_executor' in globals():
    rviz_executor.shutdown(timeout_sec=1.0)
if 'rviz_executor_thread' in globals() and rviz_executor_thread.is_alive():
    rviz_executor_thread.join(timeout=1.0)
if 'rviz_node' in globals():
    rviz_node.destroy_node()
if globals().get('rviz_owns_rclpy', False) and rclpy.ok():
    rclpy.shutdown()
print('Scene and RViz publishers stopped.')
